# [Chapter 10 — Sensitivity Analysis, §10.3] LHS-PRCC Global Sensitivity Analysis

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 10 — Sensitivity Analysis
**Considerations developed:** 10
**Estimated runtime:** ≤ 1m

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Latin Hypercube Sampling combined with Partial Rank Correlation Coefficients gives global sensitivity rankings that capture nonlinear effects and parameter ranges, complementing the local indices.

## What you should already know
Chapter 10.1-7.2 local sensitivity notebooks.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_10
from shared.seeds import set_seed_chapter_10
from shared.verification import assert_within_tolerance
set_seed_chapter_10()
book_style()


In [ ]:
# Simple LHS implementation (avoid SALib for clarity)
from scipy.stats import rankdata

def latin_hypercube(n, ranges, seed=42):
    # LHS with uniform marginals over each range
    rng = np.random.default_rng(seed)
    d = len(ranges)
    samples = np.zeros((n, d))
    for j, (lo, hi) in enumerate(ranges):
        cuts = np.linspace(0, 1, n+1)
        u = rng.uniform(0, 1, n)
        x_unit = cuts[:-1] + u / n
        rng.shuffle(x_unit)
        samples[:, j] = lo + (hi - lo) * x_unit
    return samples

# Parameter ranges (±20% around baseline)
params = baseline_chapter_10()
param_names = ['c_I', 'beta', 'tau_R']
base_vals = [params[n] for n in param_names]
ranges = [(b * 0.8, b * 1.2) for b in base_vals]

# Sample and evaluate R_0
n_samples = 1000
samples = latin_hypercube(n_samples, ranges)
R0_values = samples.prod(axis=1)  # since R_0 = c_I * beta * tau_R

# PRCC: rank-correlation between each parameter and R_0
ranks_params = np.array([rankdata(samples[:, j]) for j in range(3)]).T
ranks_R0 = rankdata(R0_values)

prcc = []
for j in range(3):
    other = list(range(3)); other.remove(j)
    # Partial correlation: regress out other params from both
    X_other = ranks_params[:, other]
    X_other_with_intercept = np.column_stack([np.ones(n_samples), X_other])
    # Residuals after regressing param j on others
    coef_j = np.linalg.lstsq(X_other_with_intercept, ranks_params[:, j], rcond=None)[0]
    resid_j = ranks_params[:, j] - X_other_with_intercept @ coef_j
    coef_R = np.linalg.lstsq(X_other_with_intercept, ranks_R0, rcond=None)[0]
    resid_R = ranks_R0 - X_other_with_intercept @ coef_R
    # Correlation of residuals
    if resid_j.std() == 0 or resid_R.std() == 0:
        prcc.append(0)
    else:
        prcc.append(np.corrcoef(resid_j, resid_R)[0, 1])

print(f"LHS-PRCC sensitivity indices (n_samples={n_samples}):")
for name, val in zip(param_names, prcc):
    print(f"  PRCC_{name} = {val:+.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
colors = [BOOK_COLORS['positive'] if v >= 0 else BOOK_COLORS['negative'] for v in prcc]
ax.barh(param_names, prcc, color=colors, alpha=0.85)
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel('PRCC')
ax.set_title('LHS-PRCC global sensitivity for R_0')
plt.tight_layout()
plt.show()
print("\nAll three PRCCs are near +1, consistent with R_0 = c_I * beta * tau_R")
print("being increasing (and roughly equally so) in each parameter.")
